# Does the text of an agent skill predict its replication on GitHub? — guided notebook

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/podorize/skillreuse/blob/main/notebooks/reproduce.ipynb)

**What this notebook is.** A companion to read *next to the paper*. Each part follows one section, says what to read
first, recomputes the claim from the repository's files and logs the comparison in a **scorecard** at the end. It
needs neither the raw GitSkills data nor the GitHub API: the dates of all copies were collected once and are
published in `data/derived/collection/`.

**The study in one paragraph.** We sampled 150,000 skills from GitSkills, dated every copy outside repositories that
collect skills in bulk, and found when each skill first appeared (T0). A skill is **copied** if another
developer copied it within W days after T0. We ask whether the text of a skill predicts that (RQ1) and which characteristics carry
the signal (RQ2). Skills are split **by author**; seven algorithms are spot-checked, three are tuned, and the final
models are evaluated **once** on a holdout of authors never seen in training.

**Paper:** [PDF](https://github.com/podorize/skillreuse/blob/main/paper/latex/main.pdf) ·
[3 Methods](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) · [4 Results](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md) · [5 Discussion](https://github.com/podorize/skillreuse/blob/main/paper/draft/05_discussion.md) ·
[Appendix A](https://github.com/podorize/skillreuse/blob/main/paper/draft/07_appendix_reproducibility.md)

| Part | Paper | What you recompute | Time |
|---|---|---|---|
| 0 | App. A | environment | ~2 min |
| 1 | §3.1–3.2, §4.1 | copies, outcome of the dating, T0 and author | seconds |
| 2 | §3.2, §3.4, §4.1 | window, eligibility, target, split by author | seconds |
| 3 | §3.5, §4.1 | preprocessing rules and first associations | seconds |
| 4 | §3.6, §4.2 | spot-check (1 repetition), model selection, decision threshold | ~1 min |
| 5 | §3.7, §4.3 | frozen models and control on the holdout, bootstrap by author | ~1 min |
| 6 | §4.4, App. A | seven robustness analyses, retrained | ~20 s |
| 7 | §3.8, §4.5 | permutation importance, stability, SHAP | ~1.5 min |
| 8 | §5, §6 | claims and scorecard | seconds |

> Cells that load the frozen models run through `!uv run skillreuse ...` in the project's **locked** environment
> (`uv.lock`), so scikit-learn versions match the saved models.

## Part 0 — Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

if not Path("src/skillreuse").exists():
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    else:
        if not Path("skillreuse").exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/podorize/skillreuse.git"], check=True)
        os.chdir("skillreuse")
print("working directory:", Path.cwd())
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

In [ ]:
!uv sync -q
!uv run skillreuse check

In [ ]:
import importlib.util, json
import numpy as np, pandas as pd
from IPython.display import Image, display

missing = [m for m in ("duckdb", "pyarrow", "sklearn") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow", "scikit-learn"], check=True)
sys.path.insert(0, str(Path("src").resolve()))
from skillreuse import study as S          # the project's own code
from skillreuse.stats import nb_test
get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 40)
R, C = Path("results"), Path("data/derived/collection")
SCORE = []

def check(part, what, paper, got, tol=0.0005, fmt="{:,.3f}"):
    ok = abs(float(paper) - float(got)) <= tol
    SCORE.append({"part": part, "check": what, "paper": fmt.format(paper), "recomputed": fmt.format(got), "ok": ok})
    print(("OK  " if ok else "DIFF"), what, "| paper", fmt.format(paper), "| recomputed", fmt.format(got))

def j(path):
    return json.loads((R / path).read_text())

def show(name, width=900):
    display(Image(filename=str(R / "figures" / name), width=width))

---
## Part 1 — Sample, dating of copies, T0 and author
> Read [§3.1–3.2 and §4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

The sample has every copy of 150,000 skills. For each copy outside aggregators (repositories with 100 or more
skills), the GitHub API gave us the first commit with exactly the skill's content and the creation date of the
repository. The arrival of a copy is the later of the two. T0 is the earliest arrival of a skill, and the author
owns the repository where that copy is.

In [ ]:
occ = pd.read_parquet(C / "sample_occurrences.parquet")
dates = pd.read_parquet(C / "dates.parquet")
print("skills:", occ.content_hash.nunique(), "| copies:", len(occ), "| outside aggregators:", (~occ.aggregator).sum())
d = dates.merge(occ[["content_hash", "repo_full_name", "path", "aggregator", "owner"]],
                on=["content_hash", "repo_full_name", "path"])
main = d[~d.aggregator]

# what happened to each copy outside aggregators, with the names of Figure 1c
names = {"datada": "dated", "repositorio_inacessivel": "repository gone", "arquivo_ausente": "file gone",
         "hash_nao_encontrado": "hash not found", "falha_de_coleta": "collection failure",
         "data_posterior_ao_retrato": "date after snapshot"}
status = main.status.map(names).value_counts()
display(pd.DataFrame({"copies": status, "share": (status / len(main)).map("{:.1%}".format)}))

dated = main[main.status == "datada"]
check("1", "skills in the population", 1_163_728, j("phase1/manifest.json")["counts"]["population_contents"], 0, "{:,.0f}")
check("1", "skills in the sample", 150_000, occ.content_hash.nunique(), 0, "{:,.0f}")
check("1", "copies in the sample", 308_256, len(occ), 0, "{:,.0f}")
check("1", "copies outside aggregators", 223_261, len(main), 0, "{:,.0f}")
check("1", "copies in aggregators", 84_995, occ.aggregator.sum(), 0, "{:,.0f}")
check("1", "dated copies", 198_579, len(dated), 0, "{:,.0f}")
check("1", "share of copies dated", 0.889, len(dated) / len(main), 0.0005)

In [ ]:
# T0, author and time to first copy by another developer, recomputed from the dated copies
dated = main[main.arrival.notna()].sort_values(["content_hash", "arrival", "repo_full_name", "path"])
origin = dated.drop_duplicates("content_hash")[["content_hash", "arrival", "owner"]].rename(
    columns={"arrival": "t0", "owner": "author"})
other = dated.merge(origin, on="content_hash").query("owner != author")
lag = (other.groupby("content_hash").arrival.min() - other.groupby("content_hash").t0.first()).dt.total_seconds() / 86400
T = pd.read_parquet("data/derived/study_table.parquet")
mine = origin.set_index("content_hash").loc[T.content_hash]
check("1", "skills with at least one dated copy", 135_670, len(origin), 0, "{:,.0f}")
check("1", "skills after the cap of 50 per author", 135_332, len(T), 0, "{:,.0f}")
check("1", "T0 identical to the study table", 1.0, (mine.t0.values == pd.to_datetime(T.t0, utc=True).values).mean(), 0)
check("1", "time to first copy identical", 1.0,
      np.isclose(lag.reindex(T.content_hash).fillna(-1).values, T.first_copy_lag_days.fillna(-1).values).mean(), 0)

---
## Part 2 — Window, eligibility, target and split
> Read [§3.2 and §3.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figure 1.

The window W is the smallest of 30, 60 and 90 days that covers at least 80% of the times to first copy, measured on
development skills observed for at least 90 days. A skill is eligible if its whole window ends before the snapshot
(9 July 2026), and it is copied if another developer copied it within W days. The split is by author, so here we
rebuild it from the hash of each author name and compare it with the study table.

In [ ]:
win = S.choose_window(S.load_frame("dev", eligible_only=False))
display(pd.Series(win["coverage"]).map("{:.1%}".format).rename("first copies within W days").rename_axis("W"))
check("2", "window W (days)", 60, win["W"], 0, "{:.0f}")
check("2", "first copies within 30 days", 0.735, win["coverage"][30], 0.0005)
check("2", "first copies within 60 days", 0.897, win["coverage"][60], 0.0005)
check("2", "median days to first copy", 10.5, win["lag_quantiles"][0.5], 0.05, "{:.1f}")
check("2", "first quartile of days to first copy", 2.6, win["lag_quantiles"][0.25], 0.05, "{:.1f}")

dev, hold = S.load_frame("dev", w=win["W"]), S.load_frame("holdout", w=win["W"])
check("2", "eligible development skills", 48_299, len(dev), 0, "{:,.0f}")
check("2", "eligible holdout skills", 20_564, len(hold), 0, "{:,.0f}")
check("2", "eligible holdout authors", 10_386, hold.owner.nunique(), 0, "{:,.0f}")
check("2", "share copied, development", 0.077, dev.replicated.mean(), 0.0005, "{:.1%}")
check("2", "share copied, holdout", 0.078, hold.replicated.mean(), 0.0005, "{:.1%}")

# the split, rebuilt from the hash of each author name (Section 3.4)
from skillreuse import cohort as K
holdout_authors = K.split_authors(T.rename(columns={"owner": "author"}))
rebuilt = np.where(T.owner.map(holdout_authors), "holdout", "dev")
check("2", "split rebuilt from the author hashes is identical", 1.0, (rebuilt == T.split.to_numpy()).mean(), 0)
dev_all, hold_all = T[T.split == "dev"], T[T.split == "holdout"]
check("2", "skills in the development set", 94_683, len(dev_all), 0, "{:,.0f}")
check("2", "authors in the development set", 42_872, dev_all.owner.nunique(), 0, "{:,.0f}")
check("2", "skills in the holdout", 40_649, len(hold_all), 0, "{:,.0f}")
check("2", "authors in the holdout", 18_373, hold_all.owner.nunique(), 0, "{:,.0f}")
check("2", "authors on both sides", 0, len(set(dev_all.owner) & set(hold_all.owner)), 0, "{:.0f}")
check("2", "most skills of one author", 50, T.groupby("owner").size().max(), 0, "{:.0f}")

# share of authors in the holdout, by number of skills per author
bands = pd.DataFrame({"n": T.groupby("owner").size(), "holdout": holdout_authors})
share = bands.groupby(K._band(bands.n).astype(str)).holdout.mean()
display(share.map("{:.1%}".format).rename("authors in the holdout").rename_axis("skills per author"))
check("2", "lowest share of authors in the holdout, by band", 0.30, share.min(), 0.005, "{:.1%}")
show("eda.png")

---
## Part 3 — Exploratory rules
> Read [§3.5](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figures 2 and 3.

The preprocessing rules come only from the development set. A column with a single value is removed. A column with
skewness above 2 gets a log transform in the models that are sensitive to scale. When two columns have Spearman
correlation above 0.95, only one of them stays in those models. Impossible values (a negative repository age) become
missing. Here we derive the rules again and compare them with the saved ones.

In [ ]:
dev_rules = S.eda_rules(dev.assign(repo_age_days_at_t0=dev.repo_age_days_at_t0.where(dev.repo_age_days_at_t0 >= 0)))
saved = j("phase3/eda_rules.json")
for k in ("constant", "skewed_log1p_nontree", "drop_collinear_nontree"):
    check("3", f"rule '{k}' identical to the saved one", 1.0, float(sorted(dev_rules[k]) == sorted(saved[k])), 0)

# 21 text columns and the repository age
check("3", "columns with skewness above 2", 22, len(dev_rules["skewed_log1p_nontree"]), 0, "{:.0f}")
check("3", "pairs of columns with Spearman above 0.95", 8, len(dev_rules["collinear_pairs"]), 0, "{:.0f}")
check("3", "columns removed for collinearity", 5, len(dev_rules["drop_collinear_nontree"]), 0, "{:.0f}")
check("3", "constant columns", 2, len(dev_rules["constant"]), 0, "{:.0f}")
print("removed for collinearity:", dev_rules["drop_collinear_nontree"])
print("constant:", dev_rules["constant"])
check("3", "skewness of the average word length", 218, dev.text_average_word_length.skew(), 0.5, "{:.0f}")
check("3", "missing values in repository language and license", 0.0002,
      dev[["repo_language", "repo_license"]].isna().mean().max(), 0.00005, "{:.2%}")

def quartile_rates(col):
    q = pd.qcut(dev[col].rank(method="first"), 4, labels=False)
    return dev.groupby(q).replicated.mean()

diversity, chars = quartile_rates("text_lexical_diversity"), quartile_rates("text_char_count")
check("3", "copied, lowest quartile of lexical diversity", 0.107, diversity.iloc[0], 0.0005)
check("3", "copied, highest quartile of lexical diversity", 0.048, diversity.iloc[-1], 0.0005)
check("3", "copied, lowest quartile of characters", 0.050, chars.iloc[0], 0.0005)
check("3", "copied, highest quartile of characters", 0.110, chars.iloc[-1], 0.0005)
front = dev.groupby(dev.frontmatter_valid.astype(int)).replicated.mean()
check("3", "copied, invalid front matter", 0.028, front.loc[0], 0.0005)
check("3", "copied, valid front matter", 0.083, front.loc[1], 0.0005)
check("3", "skills in each quartile", 12_000, len(dev) / 4, 100, "{:,.0f}")
check("3", "copied skills in each quartile, on average", 930, dev.replicated.sum() / 4, 10, "{:,.0f}")
show("eda_attributes.png"); show("eda_univariate.png", 600)

---
## Part 4 — Spot-check and model selection
> Read [§3.6](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.2](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 4 and Figure 4.

The slower cell re-runs the spot-check with one repetition (3 folds, where the paper used 3 repetitions, 9 folds), so
the values change a little. The ranking should not. The nested search is read from the result files: when all
configurations tie, the fastest wins. The last cells rebuild the decision threshold from the out-of-fold probabilities.

In [ ]:
!uv run skillreuse spot --repeats 1 --output-dir results/reproduce/spot | tail -9

In [ ]:
# the one-repetition re-run of the cell above, next to the paper (9 folds)
spot = pd.read_csv(R / "phase4/spot_summary.csv").set_index("config")
rerun = pd.read_csv(R / "reproduce/spot/spot_summary.csv").set_index("config")
display(pd.DataFrame({"re-run, 3 folds": rerun.roc_auc_mean, "paper, 9 folds": spot.roc_auc_mean}).loc[spot.index].round(3))
check("4", "same three best algorithms as in the paper", 1.0, float(set(rerun.index[:3]) == set(spot.index[:3])), 0)

# Table 4, spot-check (mean AUC over 9 folds)
table4_spot = {"hist_gb": 0.652, "logistic": 0.642, "mlp": 0.625, "naive_bayes": 0.619,
               "random_forest": 0.614, "knn": 0.574, "decision_tree": 0.531}
for algo, auc in table4_spot.items():
    check("4", f"Table 4, spot-check AUC, {algo}", auc, spot.loc[algo, "roc_auc_mean"])

# Table 4, nested search of the three best (mean AUC over 6 outer folds)
nested = pd.read_csv(R / "phase4/nested_summary.csv").set_index("config")
table4_nested = {"hist_gb|tuned": 0.657, "hist_gb|default": 0.653, "logistic|tuned": 0.646,
                 "logistic|default": 0.645, "mlp|tuned": 0.638, "mlp|default": 0.630}
for config, auc in table4_nested.items():
    check("4", f"Table 4, nested AUC, {config}", auc, nested.loc[config, "mean"])
display(nested.round(3))

# paired tests of the best configuration against the other five
tests = pd.read_csv(R / "phase4/nested_tests.csv").set_index("other")
display(tests.round(3))
lr = tests.loc["logistic|default"]
check("4", "HistGB tuned minus default logistic regression", 0.012, lr.mean_diff)
check("4", "  95% CI, lower end", -0.004, lr.ci95_low)
check("4", "  95% CI, upper end", 0.028, lr.ci95_high)
check("4", "HistGB tuned minus its own default", 0.004, tests.loc["hist_gb|default", "mean_diff"])
check("4", "smallest Holm-adjusted p", 0.19, tests.p_holm.min(), 0.005, "{:.2f}")
check("4", "comparisons significant after Holm", 0, (tests.p_holm < 0.05).sum(), 0, "{:.0f}")
w = j("phase4/winner.json")
print("winner:", w["winner"], "| tied:", w["tied"])
check("4", "winner is the default logistic regression", 1.0, float(w["winner"] == "logistic|default"), 0)

# decision threshold: every value from 0.01 to 0.99 on the out-of-fold probabilities of the text model
oof = pd.read_csv(R / "phase5/dev_oof_text.csv")
y_oof, p_oof = oof.y.to_numpy(), oof.prob.to_numpy()
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
threshold = grid[int(np.argmax([S.balanced_accuracy(y_oof, p_oof, t) for t in grid]))]
at = S.threshold_report(y_oof, p_oof, threshold)
check("4", "decision threshold", 0.07, threshold, 0, "{:.2f}")
check("4", "balanced accuracy at the threshold", 0.603, at["balanced_accuracy"])
check("4", "precision at the threshold", 0.105, at["precision"])
check("4", "recall at the threshold", 0.716, at["recall"])
# the paper gives this share as a whole percentage, so the tolerance is one point
check("4", "share of skills flagged at the threshold", 0.52, (p_oof >= threshold).mean(), 0.01, "{:.1%}")
show("model_selection.png")

---
## Part 5 — Single evaluation on the holdout (RQ1) and control
> Read [§3.7](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.3](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 5 and Figure 5.

The models were frozen with their hashes before the holdout was read. The first cell re-applies the four models in
`freeze.json` and checks that the AUCs are identical to the single evaluation. One of them is the original control, a
model of the origin repository without its popularity. The paper uses a second control, trained after the single
evaluation with the stars and forks of the repository at the snapshot (protocol deviation 3). Its two models are in
`freeze_control.json` and are re-applied in the next cell. The last cell repeats the bootstrap by author, which takes
about 30 seconds.

In [ ]:
!uv run skillreuse reproduce

In [ ]:
print("matches single evaluation:", j("reproduce/holdout_check.json")["matches"])

hold = S.load_frame("holdout")
y, owners = hold.replicated.to_numpy(), hold.owner.astype(str).to_numpy()
fz, fc = S.frozen(), j("phase5/freeze_control.json")
names = {"text": "text", "size": "size only", "origin_pop": "origin repository", "origin_pop+text": "origin repository + text"}

# predictions of the frozen models; the two control models with popularity are in freeze_control.json
import joblib
preds = {v: S.predict(fz, v, hold) for v in ("text", "size")}
for v, e in fc["models"].items():
    m = joblib.load(e["file"])
    preds[v] = m["pipeline"].predict_proba(hold[m["columns"]])[:, 1]
metrics = {v: S.clf_metrics(y, pv) for v, pv in preds.items()}

# Table 5, development columns: mean AUC and standard deviation over 9 folds
dev_views = pd.read_csv(R / "phase5/dev_views_summary.csv").set_index("config")
dev_auc = {v: (dev_views.loc[v, "roc_auc_mean"], dev_views.loc[v, "roc_auc_std"]) for v in ("text", "size")}
dev_auc.update({v: (fc["dev_auc"][v]["mean"], fc["dev_auc"][v]["std"]) for v in fc["dev_auc"]})
table5_dev = {"text": (0.642, 0.014), "size": (0.596, 0.018), "origin_pop": (0.676, 0.016), "origin_pop+text": (0.725, 0.012)}
for v, (mean, std) in table5_dev.items():
    check("5", f"Table 5, development AUC, {names[v]}", mean, dev_auc[v][0])
    check("5", f"Table 5, development standard deviation, {names[v]}", std, dev_auc[v][1])

# Table 5, holdout columns
check("5", "holdout skills", 20_564, len(hold), 0, "{:,.0f}")
check("5", "holdout authors", 10_386, hold.owner.nunique(), 0, "{:,.0f}")
for v, auc in {"text": 0.645, "size": 0.595, "origin_pop": 0.685, "origin_pop+text": 0.734}.items():
    check("5", f"Table 5, holdout AUC, {names[v]}", auc, metrics[v]["roc_auc"])
for v, auc in {"text": 0.130, "size": 0.106, "origin_pop": 0.199, "origin_pop+text": 0.224}.items():
    check("5", f"Table 5, PR AUC, {names[v]}", auc, metrics[v]["pr_auc"])
check("5", "text minus size", 0.050, metrics["text"]["roc_auc"] - metrics["size"]["roc_auc"])
check("5", "gain of the text over the origin repository", 0.049,
      metrics["origin_pop+text"]["roc_auc"] - metrics["origin_pop"]["roc_auc"])
check("5", "PR AUC of the text model, times the share copied", 1.7, metrics["text"]["pr_auc"] / y.mean(), 0.05, "{:.1f}")
check("5", "Brier score of the text model", 0.0701, metrics["text"]["brier"], 0.00005, "{:.4f}")
check("5", "Brier score of predicting the share copied for every skill", 0.0715, y.mean() * (1 - y.mean()), 0.00005, "{:.4f}")

# what the threshold does on the holdout (Figure 5c)
at = S.threshold_report(y, preds["text"], fz["threshold"])
flagged = at["true_positive"] + at["false_positive"]
true_negative = len(y) - flagged - at["false_negative"]
display(pd.DataFrame([[true_negative, at["false_positive"]], [at["false_negative"], at["true_positive"]]],
                     index=["not copied", "copied"], columns=["predicted not copied", "predicted copied"]))
check("5", "skills flagged at the threshold", 10_761, flagged, 0, "{:,.0f}")
check("5", "copied skills in the holdout", 1_594, y.sum(), 0, "{:,.0f}")
check("5", "copied skills among the flagged", 1_137, at["true_positive"], 0, "{:,.0f}")
check("5", "copied skills not flagged", 457, at["false_negative"], 0, "{:,.0f}")
check("5", "not copied skills flagged", 9_624, at["false_positive"], 0, "{:,.0f}")
check("5", "not copied skills not flagged", 9_346, true_negative, 0, "{:,.0f}")
check("5", "precision at the threshold", 0.106, at["precision"])
check("5", "recall at the threshold", 0.71, at["recall"], 0.005, "{:.0%}")
check("5", "balanced accuracy at the threshold", 0.603, at["balanced_accuracy"])
show("results.png")

In [ ]:
# bootstrap by author: 2,000 resamples of whole authors with a fixed seed, about 30 seconds
from sklearn.metrics import roc_auc_score
from skillreuse.stats import owner_bootstrap, boot_p, holm

boot = {v: owner_bootstrap(owners, lambda w, pv=pv: roc_auc_score(y, pv, sample_weight=w)) for v, pv in preds.items()}
interval = lambda samples: np.percentile(samples, [2.5, 97.5])
gain = boot["origin_pop+text"] - boot["origin_pop"]
over_size = boot["text"] - boot["size"]

paper_ci = {"text": (0.624, 0.666), "size": (0.573, 0.615), "origin_pop": (0.664, 0.704), "origin_pop+text": (0.714, 0.753)}
for v, (low, high) in paper_ci.items():
    check("5", f"95% CI of the AUC, {names[v]}, lower end", low, interval(boot[v])[0])
    check("5", f"95% CI of the AUC, {names[v]}, upper end", high, interval(boot[v])[1])
check("5", "95% CI of text minus size, lower end", 0.033, interval(over_size)[0])
check("5", "95% CI of text minus size, upper end", 0.069, interval(over_size)[1])
check("5", "95% CI of the gain of the text, lower end", 0.035, interval(gain)[0])
check("5", "95% CI of the gain of the text, upper end", 0.063, interval(gain)[1])
p_h1, p_control = holm([boot_p(boot["text"], 0.5), boot_p(gain, 0.0)])
check("5", "Holm-adjusted p, text model above 0.5", 0.002, p_h1, 0.0005, "{:.3f}")
check("5", "Holm-adjusted p, gain of the text", 0.002, p_control, 0.0005, "{:.3f}")

---
## Part 6 — Robustness
> Read [§4.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 6, and [Appendix A](https://github.com/podorize/skillreuse/blob/main/paper/draft/07_appendix_reproducibility.md) for the last two analyses.

Each analysis changes one choice of ours and measures the text model again on the holdout. Changing the window, the
definition of aggregator or the period retrains the model on the development set with the selected configuration
(logistic regression, default settings). The other two, keeping only skills whose copies could all be dated and only
skills whose names do not appear in development, use the frozen model. The code cell is slower (about half a minute)
because it retrains and bootstraps every analysis.

In [ ]:
# every analysis is recomputed with the study's own code; nothing is written to results/
fz, rules, W = S.frozen(), S.load_rules(), S.window()
dev, hold = S.load_frame("dev"), S.load_frame("holdout")
y, owners = hold.replicated.to_numpy(), hold.owner.astype(str).to_numpy()

analyses = {
    "window_30": S._retrain_eval(S.load_frame("dev", 30), S.load_frame("holdout", 30), fz, rules),
    "window_90": S._retrain_eval(S.load_frame("dev", 90), S.load_frame("holdout", 90), fz, rules),
    "aggregators_agg500": S._retrain_eval(S.load_frame("dev", W, "_agg500"), S.load_frame("holdout", W, "_agg500"), fz, rules),
    "aggregators_aggnone": S._retrain_eval(S.load_frame("dev", W, "_aggnone"), S.load_frame("holdout", W, "_aggnone"), fz, rules),
}
# temporal split: train on the earlier 70% of the development skills, test on the holdout skills that appeared later
cutoff = dev.t0_used.quantile(0.70)
analyses["temporal"] = S._retrain_eval(dev[dev.t0_used < cutoff], hold[hold.t0_used >= cutoff], fz, rules)

# the frozen text model, on a part of the holdout
p_text = S.predict(fz, "text", hold)
dated = hold.all_copies_dated.to_numpy(dtype=bool)
analyses["all_copies_dated"] = S._auc_ci(y[dated], p_text[dated], owners[dated])
names_in_dev = set(dev.loc[dev.name_norm != "", "name_norm"])
unseen = ~(hold.name_norm.ne("") & hold.name_norm.isin(names_in_dev)).to_numpy()
analyses["name_unseen"] = {"share": unseen.mean(), **S._auc_ci(y[unseen], p_text[unseen], owners[unseen])}
display(pd.DataFrame(analyses).T[["rows", "prevalence", "auc", "ci95"]])

# Table 6: skills, share copied, AUC and the two ends of the 95% interval
table6 = {"window_30": (28_464, 0.057, 0.648, 0.628, 0.666), "window_90": (13_460, 0.094, 0.633, 0.612, 0.656),
          "aggregators_agg500": (20_648, 0.090, 0.644, 0.624, 0.663), "aggregators_aggnone": (20_694, 0.125, 0.653, 0.635, 0.671),
          "temporal": (6_031, 0.067, 0.634, 0.589, 0.681)}
label = {"window_30": "window of 30 days", "window_90": "window of 90 days", "aggregators_agg500": "aggregators, limit of 500",
         "aggregators_aggnone": "aggregators, no exclusion", "temporal": "temporal split"}
for key, (skills, share, auc, low, high) in table6.items():
    a = analyses[key]
    check("6", f"Table 6, {label[key]}, skills", skills, a["rows"], 0, "{:,.0f}")
    check("6", f"Table 6, {label[key]}, share copied", share, a["prevalence"], 0.0005, "{:.1%}")
    check("6", f"Table 6, {label[key]}, AUC", auc, a["auc"])
    check("6", f"Table 6, {label[key]}, 95% CI, lower end", low, a["ci95"][0])
    check("6", f"Table 6, {label[key]}, 95% CI, upper end", high, a["ci95"][1])
check("6", "cutoff of the temporal split (year, month, day)", 20260414, int(cutoff.strftime("%Y%m%d")), 0, "{:.0f}")

# Appendix A: the two further checks
a = analyses["all_copies_dated"]
check("6", "Appendix A, only skills with every copy dated, skills", 19_862, a["rows"], 0, "{:,.0f}")
check("6", "Appendix A, only skills with every copy dated, share copied", 0.055, a["prevalence"], 0.0005, "{:.1%}")
check("6", "Appendix A, only skills with every copy dated, AUC", 0.626, a["auc"])
check("6", "Appendix A, only skills with every copy dated, 95% CI, lower end", 0.600, a["ci95"][0])
check("6", "Appendix A, only skills with every copy dated, 95% CI, upper end", 0.649, a["ci95"][1])
a = analyses["name_unseen"]
check("6", "Appendix A, only unseen names, skills", 15_154, a["rows"], 0, "{:,.0f}")
check("6", "Appendix A, only unseen names, share of the holdout", 0.737, a["share"], 0.0005, "{:.1%}")
check("6", "Appendix A, only unseen names, AUC", 0.622, a["auc"])
check("6", "Appendix A, only unseen names, 95% CI, lower end", 0.593, a["ci95"][0])
check("6", "Appendix A, only unseen names, 95% CI, upper end", 0.651, a["ci95"][1])

# what the text of Section 4.4 says about the five alternatives
alternatives = [analyses[k] for k in table6]
main_auc = j("phase6/holdout_results.json")["H1"]["auc"]
check("6", "lowest AUC of the five alternatives", 0.633, min(a["auc"] for a in alternatives))
check("6", "highest AUC of the five alternatives", 0.653, max(a["auc"] for a in alternatives))
check("6", "every 95% interval lies above 0.5", 1.0, float(all(a["ci95"][0] > 0.5 for a in alternatives)), 0)
check("6", "largest distance of a window AUC from the main result", 0.012,
      max(abs(analyses[k]["auc"] - main_auc) for k in ("window_30", "window_90")), 0.0005)
check("6", "copied skills with no exclusion of aggregators, over the main analysis, is above 1.5", 1.0,
      float(analyses["aggregators_aggnone"]["prevalence"] / y.mean() > 1.5), 0)

---
## Part 7 — What the model learned (RQ2)
> Read [§3.8](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.5](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figures 6 and 7.

The model interpreted is the frozen text model, a logistic regression. Permutation importance shuffles an attribute,
or a whole family of attributes at once, and measures how much the AUC drops. SHAP splits each prediction into the
contribution of each attribute. Both are computed on 20,000 holdout skills made of whole authors. The code cell is
slower (about two minutes), mostly because of the bootstrap intervals of the families and the nine development folds
of the stability check.

In [ ]:
import joblib, shap
fz, rules = S.frozen(), S.load_rules()
saved_model = joblib.load(fz["models"]["text"]["file"])
pipe, cols = saved_model["pipeline"], saved_model["columns"]
sample = S.interp_sample(S.load_frame("holdout"))          # 20,000 skills made of whole authors
y_s, owners_s = sample.replicated.to_numpy(), sample.owner.astype(str).to_numpy()
check("7", "skills in the interpretation sample", 20_000, len(sample), 0, "{:,.0f}")

# permutation importance: the families with 95% intervals (200 resamples of authors), the attributes without
families = S.permutation_table(pipe, cols, sample, y_s, owners_s, S.SEED, S.N_BOOT_PERM, families_only=True).set_index("feature")
attributes = S.permutation_table(pipe, cols, sample, y_s, owners_s, S.SEED, n_boot=1)
attributes = attributes[~attributes.feature.str.startswith("family:")].set_index("feature").auc_drop.sort_values(ascending=False)
display(families.round(4))

paper_families = {"Text style": 0.092, "Size": 0.047, "Front matter": 0.035, "Links": 0.022, "Code": 0.020, "Markdown structure": 0.012}
for family, drop in paper_families.items():
    check("7", f"Section 4.5, AUC drop, {family} family", drop, families.loc[f"family:{family}", "auc_drop"])
check("7", "order of the families by importance", 1.0, float(list(families.index) == [f"family:{f}" for f in paper_families]), 0)
check("7", "AUC drop, text style, lower end of the 95% interval", 0.071, families.loc["family:Text style", "ci95_low"])
check("7", "AUC drop, text style, upper end of the 95% interval", 0.109, families.loc["family:Text style", "ci95_high"])
check("7", "AUC drop, number of sentences", 0.074, attributes["text_sentence_count"])
top5 = ["text_sentence_count", "text_lexical_diversity", "text_average_sentence_words", "text_average_word_length", "frontmatter_valid"]
check("7", "five most important attributes, in this order", 1.0, float(list(attributes.index[:5]) == top5), 0)
check("7", "the whole size family matters less than the number of sentences alone", 1.0,
      float(families.loc["family:Size", "auc_drop"] < attributes["text_sentence_count"]), 0)

# stability: the same permutation on the validation part of each of the nine development folds
dev_frame, fp = S.load_frame("dev"), j("phase5/final_params.json")
y_dev, by_fold = dev_frame.replicated.to_numpy(), {}
for r, f, train, valid_idx in S.folds(dev_frame):
    fold_pipe, fold_cols = S.pipeline(fp["algo"], "text", rules, fp["views"]["text"])
    fold_pipe.fit(dev_frame.iloc[train][fold_cols], y_dev[train])
    valid = dev_frame.iloc[valid_idx].reset_index(drop=True)
    drops = S.permutation_table(fold_pipe, fold_cols, valid, y_dev[valid_idx], valid.owner.astype(str).to_numpy(),
                                S.SEED + 10 * r + f, n_boot=1, families_only=True)
    by_fold[f"r{r}f{f}"] = drops.set_index("feature").auc_drop
stability = pd.DataFrame(by_fold)
stability["holdout"] = families.auc_drop
check("7", "folds in which text style is the first family", 9, int((stability.drop(columns="holdout").idxmax() == "family:Text style").sum()), 0, "{:.0f}")
rho = stability.rank().corr("spearman")["holdout"].drop("holdout")
check("7", "lowest Spearman correlation of a fold with the holdout", 0.71, rho.min(), 0.005, "{:.2f}")
check("7", "highest Spearman correlation of a fold with the holdout", 1.00, rho.max(), 0.005, "{:.2f}")

# SHAP values of the logistic regression, in log-odds
x_t = pipe[:-1].transform(sample[cols])
shap_values = shap.LinearExplainer(pipe.named_steps["model"], shap.maskers.Independent(x_t, max_samples=len(x_t))).shap_values(x_t)
feature_names = list(pipe.named_steps["pre"].get_feature_names_out())
position = {c: next(k for k, n in enumerate(feature_names) if n.endswith(f"__{c}")) for c in cols}
mean_abs = pd.Series({c: np.abs(shap_values[:, k]).mean() for c, k in position.items()}).sort_values(ascending=False)
check("7", "attributes in the top five of both rankings, permutation and SHAP", 4, len(set(mean_abs.index[:5]) & set(top5)), 0, "{:.0f}")

def direction(c):
    return np.sign(pd.Series(sample[c].to_numpy(dtype=float)).corr(pd.Series(shap_values[:, position[c]]), method="spearman"))
up = ["text_sentence_count", "text_average_sentence_words", "text_average_word_length", "frontmatter_valid"]
down = ["text_lexical_diversity", "frontmatter_description_missing"]
check("7", "sentences, words per sentence, word length and a valid front matter push toward copied", 1.0, float(all(direction(c) > 0 for c in up)), 0)
check("7", "a varied vocabulary and a missing description push away", 1.0, float(all(direction(c) < 0 for c in down)), 0)
is_valid = sample.frontmatter_valid.to_numpy() == 1
gap = shap_values[is_valid, position["frontmatter_valid"]].mean() - shap_values[~is_valid, position["frontmatter_valid"]].mean()
check("7", "invalid front matter, log-odds below a valid one", 0.9, gap, 0.05, "{:.1f}")

# four attributes whose direction flips between the simple tables and the model
flips = ["text_char_count", "frontmatter_description_char_count", "markdown_heading_count", "markdown_inline_code_count"]
rates = pd.read_csv(R / "phase3/univariate_rates.csv").set_index("feature")
coef = pd.Series(pipe.named_steps["model"].coef_[0], index=feature_names)
check("7", "alone, the top quartile of these four attributes has more copied skills", 1.0, float(all(rates.loc[c, "q4"] > rates.loc[c, "q1"] for c in flips)), 0)
check("7", "in the model, the same four attributes have a negative coefficient", 1.0,
      float(all(coef[[n for n in feature_names if n.endswith("__" + c)][0]] < 0 for c in flips)), 0)
show("importance.png"); show("shap.png")

---
## Part 8 — What may be claimed
> Read [§5.1, §5.2 and §5.3](https://github.com/podorize/skillreuse/blob/main/paper/draft/05_discussion.md) and [§6](https://github.com/podorize/skillreuse/blob/main/paper/draft/06_conclusion.md).

Each row below is a claim the paper makes, or refuses to make, with the section where it appears and the number this
notebook recomputed for it. The scorecard at the end counts every check of the notebook.

Not recomputed here: the numbers of the whole GitSkills dataset (3,797,117 files, 1,877,981 distinct skills, 2,867
aggregators, the 38% of skills found only in aggregators) and the dating of the copies through the GitHub API. They
need the raw data and the API, which are levels 3 and 4 of the README.

In [ ]:
# two numbers of the paper that no earlier part checked
check("8", "attributes of the text model", 22, len(joblib.load(S.frozen()["models"]["text"]["file"])["columns"]), 0, "{:.0f}")
check("8", "copies we tried to date whose repository was gone", 0.094, (main.status == "repositorio_inacessivel").mean(), 0.0005, "{:.1%}")

def value(part, text):
    """The recomputed value of a check, as it was printed (the last time it ran)."""
    rows = [r for r in SCORE if r["part"] == part and r["check"] == text]
    return rows[-1]["recomputed"]

claims = pd.DataFrame([
    ("supported", "The text at T0 predicts copying by other developers for unseen authors", "§4.3, §6",
     f"AUC {value('5', 'Table 5, holdout AUC, text')} [{value('5', '95% CI of the AUC, text, lower end')}; {value('5', '95% CI of the AUC, text, upper end')}]"),
    ("supported", "It says more than the text length and more than the origin repository, popularity included", "§4.3",
     f"{value('5', 'text minus size')} AUC above the size-only model and {value('5', 'gain of the text over the origin repository')} above the origin repository"),
    ("supported", "It holds for future skills, other windows and other definitions of aggregator", "§4.4, §6",
     f"AUC from {value('6', 'lowest AUC of the five alternatives')} to {value('6', 'highest AUC of the five alternatives')} in the five alternatives, {value('6', 'Table 6, temporal split, AUC')} on future skills"),
    ("supported", "Writing style is the most important family, in every fold", "§4.5",
     f"first in {value('7', 'folds in which text style is the first family')} of 9 folds, AUC drop of {value('7', 'Section 4.5, AUC drop, Text style family')}"),
    ("not supported", "That writing style causes copying", "§5.3", "observational study: associations only"),
    ("not supported", "That the model can flag single skills reliably", "§4.3, §5.2",
     f"precision of {float(value('5', 'precision at the threshold')):.1%} at the threshold"),
    ("not supported", "Anything about copies deleted before July 2026", "§5.3",
     f"{value('8', 'copies we tried to date whose repository was gone')} of the copies we tried to date had disappeared since"),
    ("not supported", "The order of the families after the first", "§4.5",
     f"Spearman correlation of a fold with the holdout as low as {value('7', 'lowest Spearman correlation of a fold with the holdout')}"),
], columns=["", "Claim", "Paper", "Recomputed in this notebook"])
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(claims)

# scorecard
sc = pd.DataFrame(SCORE)
sc["result"] = sc.ok.map({True: "OK", False: "DIFF"})
display(pd.crosstab(sc.part, sc.result).reindex(columns=["OK", "DIFF"], fill_value=0))
print(f"{(sc.result == 'OK').sum()}/{len(sc)} checks within tolerance")
if (sc.result == "DIFF").any():
    display(sc.loc[sc.result == "DIFF", ["part", "check", "paper", "recomputed"]])
else:
    print("no check differs from the paper")